# Eksplorasi: Membandingkan Waktu & Jawaban Antar Model

Di langkah 08 ini kamu akan membandingkan **beberapa model berbeda** di server Ollama kampus dengan **prompt yang sama persis**, lalu mengamati:

1. **Waktu respons** (latency) — berapa detik tiap model butuh untuk menjawab.
2. **Isi jawaban** — apakah gaya, panjang, dan akurasinya berbeda?

Ini membuktikan konsep dari materi: *respons model bersifat stokastik dan kemampuan tiap model berbeda* — sehingga prompt engineering perlu disesuaikan dengan model yang dipakai.

> Pastikan `pip install -r requirements.txt` sudah dijalankan dan pilih kernel Python dari `.venv`.

In [1]:
import time

import requests

BASE_URL = "https://ollama.if.unismuh.ac.id"

# Lihat model apa saja yang tersedia di server
model_list = [m["name"] for m in requests.get(f"{BASE_URL}/api/tags", timeout=10).json()["models"]]
print("Model tersedia:")
for m in model_list:
    print(" -", m)

/Users/yasser/Latihan/generative-ai-for-beginners/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Model tersedia:
 - hf.co/gmonsoon/gemma2-9b-cpt-sahabatai-v1-instruct-GGUF:Q8_0
 - gpt-oss:latest
 - gemma4-16k:latest
 - qwen2.5:7b-instruct
 - qwen2.5-coder:32b
 - bge-m3:latest
 - gemma4:latest
 - gemma3:27b
 - qwen3.6:latest
 - translategemma:latest
 - nomic-embed-text:latest
 - qwen3-coder:30b
 - phi4-reasoning:plus
 - qwen2.5vl:latest
 - smollm2:135m
 - glm-4.6:cloud
 - llama3.2:latest


## Fungsi bantu: kirim prompt + ukur waktu

Kita membungkus pemanggilan `/api/generate` dengan pengukuran waktu memakai `time.perf_counter()`.

In [3]:
def tanya(model: str, prompt: str) -> tuple:
    """Kembalikan (jawaban, durasi_detik) dari satu model."""
    mulai = time.perf_counter()
    r = requests.post(
        f"{BASE_URL}/api/generate",
        json={"model": model, "prompt": prompt, "stream": False},
        timeout=300,
    )
    r.raise_for_status()
    durasi = time.perf_counter() - mulai
    return r.json()["response"], durasi

## Bandingkan model dengan prompt yang sama

Ubah `PROMPT` sesukamu, dan ubah `model_uji` jika ingin memilih model tertentu (default: maksimal 3 model pertama agar tidak terlalu lama).

In [4]:
PROMPT = "Jelaskan apa itu fotosintesis dalam 2 kalimat sederhana."
model_uji = model_list[:3]

hasil = []
for model in model_uji:
    print(f"Menanya {model} ...")
    jawaban, durasi = tanya(model, PROMPT)
    hasil.append((model, durasi, jawaban))

print("\n" + "=" * 70)
for model, durasi, jawaban in hasil:
    print(f"\n[{model}] — {durasi:.2f} detik")
    print(jawaban.strip()[:500])
    print("-" * 70)

Menanya hf.co/gmonsoon/gemma2-9b-cpt-sahabatai-v1-instruct-GGUF:Q8_0 ...
Menanya gpt-oss:latest ...
Menanya gemma4-16k:latest ...


[hf.co/gmonsoon/gemma2-9b-cpt-sahabatai-v1-instruct-GGUF:Q8_0] — 109.49 detik
Fotosintesis adalah proses yang dilakukan oleh tumbuhan untuk mengubah energi cahaya matahari menjadi energi kimia dalam bentuk glukosa, makanan bagi tumbuhan tersebut. Proses ini memerlukan air dan karbondioksida sebagai bahan baku dan menghasilkan oksigen sebagai produk sampingan.
----------------------------------------------------------------------

[gpt-oss:latest] — 7.69 detik
Fotosintesis adalah proses di mana tumbuhan, alga, dan beberapa bakteri mengubah cahaya matahari menjadi energi kimia dalam bentuk glukosa. Selama proses ini, CO₂ diambil dari udara dan O₂ dilepaskan sebagai produk samping.
----------------------------------------------------------------------

[gemma4-16k:latest] — 32.87 detik
Fotosintesis adalah proses alami yang dilakukan tumbuhan untuk membuat mak

## Bonus: model yang sama, prompt yang sama, dua kali

Jalankan sel di bawah untuk membuktikan bahwa **model yang sama pun bisa memberi jawaban berbeda** untuk prompt yang identik (sifat *stokastik*).

In [5]:
model = model_uji[0]
for percobaan in (1, 2):
    jawaban, durasi = tanya(model, PROMPT)
    print(f"Percobaan {percobaan} ({durasi:.2f} dtk): {jawaban.strip()[:200]}\n")

Percobaan 1 (5.66 dtk): Fotosintesis adalah proses di mana tumbuhan, alga, dan beberapa bakteri mengubah energi cahaya matahari menjadi energi kimia yang disimpan dalam bentuk gula (glukos). Proses ini menggunakan air dan ka

Percobaan 2 (70.27 dtk): Fotosintesis adalah proses di mana tumbuhan menggunakan energi cahaya matahari, air, dan karbon dioksida untuk membuat makanan mereka sendiri (gula) dan menghasilkan oksigen sebagai produk sampingan. 



## Pertanyaan Refleksi (tulis jawabanmu di sel ini)

1. Model mana yang paling cepat? Apakah yang paling cepat juga yang jawabannya paling baik?
2. Apa perbedaan gaya jawaban antar model untuk prompt yang sama?
3. Pada percobaan "dua kali", apakah jawabannya identik? Mengapa hal ini penting diperhatikan saat membangun aplikasi?

**Jawaban saya:**

1. ...
2. ...
3. ...